# Second 1.0 — Stage T2: Engine-Grounded Teacher Synthesis

This notebook runs Stage T2 teacher synthesis and rejection filtering on Google Colab with an **NVIDIA L4 or A100 GPU**.
It implements Stage T2 of `TRAINING_PLAN.md` using the winning candidate model: **`Qwen/Qwen3.5-27B`** (Apache 2.0).

### Pipeline Stages in this Notebook
1. **Stage T2.3**: Verify 8,000 taxonomy-stratified teacher candidate prompts (`data/processed/teacher_prompts.jsonl`).
2. **Stage T2.2 / T2.3**: Run batched teacher synthesis with `Qwen/Qwen3.5-27B` in 4-bit BitsAndBytes quantization.
3. **Stage T2.4**: Run multi-sample rejection filtering using Stage T1 reward verifier and acceptance gates.
4. **Stage T2.5**: Rescue gold human data from `sft_train.jsonl` and apply MinHash / 5-word prefix diversity deduplication.
5. **Final Output**: Save `data/processed/sft_v2_train.jsonl` to Google Drive for Stage T3 SFT training.

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 or A100 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Active GPU: {device_name} ({vram_gb:.1f} GB VRAM)")

In [ ]:
# Step 2: Mount Google Drive for Persistent Storage
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive/second")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Google Drive mounted. Storage directory: {DRIVE_DIR}")

In [ ]:
# Step 3: Extract Stage T2 Teacher Bundle or Setup Local Repo
import os
import zipfile

WORK_DIR = Path("/content/Second")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    DRIVE_DIR / "data" / "phase2_teacher_bundle.zip",
    Path("/content/phase2_teacher_bundle.zip"),
    Path("data/processed/phase2_teacher_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    os.chdir(str(WORK_DIR))
    print(f"Bundle extracted. Working directory: {os.getcwd()}")
elif Path("data/pipeline/synthesize_teacher.py").is_file():
    print("Working in existing repository checkout.")
else:
    raise FileNotFoundError("phase2_teacher_bundle.zip not found. Please upload it to Google Drive or /content.")

In [ ]:
# Step 4: Install Dependencies
!pip install -q transformers accelerate bitsandbytes chess langdetect pandas

In [ ]:
# Step 5: Verify Teacher Prompts (8,000 items, zero contamination)
from data.pipeline.build_teacher_prompts import verify_teacher_prompts
import json

prompts_file = Path("data/processed/teacher_prompts.jsonl")
with open(prompts_file, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

summary = verify_teacher_prompts(records)
print(f"Verified {len(records)} teacher prompts with zero devset/testset contamination.")
print(f"Tasks: {summary['task_distribution']}")
print(f"Details: {summary['detail_distribution']}")
print(f"Levels: {summary['level_distribution']}")
print(f"Motif coverage: {summary['motif_coverage']}")

In [ ]:
# Step 6: Run Batched Teacher Synthesis with Qwen/Qwen3.5-27B (4-bit)
!python -m data.pipeline.synthesize_teacher \
    --prompts data/processed/teacher_prompts.jsonl \
    --output data/processed/teacher_completions.jsonl \
    --model Qwen/Qwen3.5-27B \
    --load-in-4bit \
    --batch-size 4 \
    --max-new-tokens 512 \
    --samples-per-prompt 1

In [ ]:
# Backup raw completions to Google Drive
import shutil
drive_comp_path = DRIVE_DIR / "data" / "teacher_completions.jsonl"
drive_comp_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2("data/processed/teacher_completions.jsonl", drive_comp_path)
print(f"Backed up teacher completions to {drive_comp_path}")

In [ ]:
# Step 7: Run Multi-Sample Rejection Filtering (Stage T2.4)
!python -m data.pipeline.filter_teacher \
    --prompts data/processed/teacher_prompts.jsonl \
    --completions data/processed/teacher_completions.jsonl \
    --output-dir data/processed \
    --max-candidates 4

In [ ]:
# Step 8: Diversity Pruning and Human Gold Data Rescue (Stage T2.5)
!python -m data.pipeline.dedupe_diversity \
    --teacher-data data/processed/teacher_verified_non_thinking.jsonl \
    --human-data data/processed/sft_train.jsonl \
    --output data/processed/sft_v2_train.jsonl \
    --max-prefix-pct 0.015

In [ ]:
# Step 9: Save Final SFT v2 Dataset to Google Drive and Print Summary
final_path = DRIVE_DIR / "data" / "sft_v2_train.jsonl"
shutil.copy2("data/processed/sft_v2_train.jsonl", final_path)

with open("data/processed/sft_v2_train.jsonl", "r", encoding="utf-8") as f:
    sft_v2_records = [json.loads(line) for line in f if line.strip()]

print(f"\nStage T2 Complete!")
print(f"Final SFT v2 Dataset: {len(sft_v2_records)} gold verified examples.")
print(f"Saved to persistent Google Drive: {final_path}")
print("Ready to proceed to Stage T3: SFT v2 Iteration and Capacity Ablations.")